# Export Stock Data to Azure

Exports transformed stock data from the stock_daily_data table to Azure Data Lake Storage in Parquet format

In [0]:
# Create widgets for configuration
dbutils.widgets.text("environment", "dev", "Environment")
dbutils.widgets.text("azure_path", "abfss://container@storageaccount.dfs.core.windows.net/stock_data", "Azure Storage Path")

# Get configuration values
environment = dbutils.widgets.get("environment")
azure_path = dbutils.widgets.get("azure_path")

# Print configuration
print(f"Environment: {environment}")
print(f"Azure Storage Path: {azure_path}")
print(f"Source Table: {environment}.stock_daily_data")

Environment: dev
Azure Storage Path: abfss://container@storageaccount.dfs.core.windows.net/stock_data
Source Table: dev.stock_daily_data


In [0]:
# Get widget values
environment = dbutils.widgets.get("environment")

# Read data from the stock_daily_data table
table_name = f"{environment}.stock_daily_data"

print(f"Reading data from {table_name}...")
df = spark.table(table_name)

# Show basic info
row_count = df.count()
print(f"Total rows to export: {row_count:,}")
print(f"\nSchema:")
df.printSchema()

Reading data from dev.stock_daily_data...
Total rows to export: 102

Schema:
root
 |-- Stock: string (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- fetch_date: date (nullable = true)



In [0]:
# Get widget values
azure_path = dbutils.widgets.get("azure_path")

# Export data to Azure Data Lake Storage in Parquet format
try:
    print(f"Writing data to {azure_path}...")
    
    # Write to Azure with partitioning by Stock symbol
    df.write \
        .mode("overwrite") \
        .partitionBy("Stock") \
        .parquet(azure_path)
    
    print(f"\n✓ Successfully exported {row_count:,} rows to Azure Storage")
    print(f"  Location: {azure_path}")
    print(f"  Format: Parquet")
    print(f"  Partitioned by: Stock")
    
except Exception as e:
    print(f"\n✗ Error exporting data to Azure: {str(e)}")
    print(f"\nNote: This error is expected if Azure credentials are not configured.")
    print(f"To configure Azure access, set up:")
    print(f"  - Service principal credentials, OR")
    print(f"  - Storage account access key, OR")
    print(f"  - SAS token")
    raise

Writing data to abfss://container@storageaccount.dfs.core.windows.net/stock_data...

✗ Error exporting data to Azure: (com.databricks.common.filesystem.SparkKeyProviderException) Failure to initialize configuration for storage account storageaccount.dfs.core.windows.net: Invalid configuration value detected for fs.azure.account.key

JVM stacktrace:
com.databricks.common.filesystem.SparkKeyProviderException
	at com.databricks.common.filesystem.LokiABFS.initialize(LokiABFS.scala:48)
	at com.databricks.sql.io.LokiFileSystem$.$anonfun$getLokiFS$3(LokiFileSystem.scala:137)
	at com.databricks.spark.util.FrameProfiler$.$anonfun$record$1(FrameProfiler.scala:114)
	at com.databricks.spark.util.FrameProfilerExporter$.maybeExportFrameProfiler(FrameProfilerExporter.scala:201)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:105)
	at com.databricks.sql.io.LokiFileSystem$.$anonfun$getLokiFS$2(LokiFileSystem.scala:133)
	at com.databricks.sql.io.FileSystemCache.getOrCompute(FileS

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-5237173320587452>, line 12
      6 print(f"Writing data to {azure_path}...")
      8 # Write to Azure with partitioning by Stock symbol
      9 df.write \
     10     .mode("overwrite") \
     11     .partitionBy("Stock") \
---> 12     .parquet(azure_path)
     14 print(f"\n✓ Successfully exported {row_count:,} rows to Azure Storage")
     15 print(f"  Location: {azure_path}")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:779, in DataFrameWriter.parquet(self, path, mode, partitionBy, compression)
    777     self.partitionBy(partitionBy)
    778 self._set_opts(compression=compression)
--> 779 self.format("parquet").save(path)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **